In [45]:
from langgraph.graph import StateGraph, START, END, add_messages
from langgraph.prebuilt import ToolNode, tools_condition # toolNode and toolCondition
from langgraph.checkpoint.memory import InMemorySaver # checkpointers

from langchain_openai import ChatOpenAI
from typing import TypedDict, Annotated, Literal
from langchain_core.messages import HumanMessage, BaseMessage

# tool related imports
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchResults
from dotenv import load_dotenv




In [46]:
"""
Task: 

- Langgraph with tool node

"""

load_dotenv()

True

In [47]:
class ToolsState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [48]:
# creating a tool

# tool - 1
web_search_tool = DuckDuckGoSearchResults()


# tool - 2
@tool
def calculator(firstNumber: float, secondNumber: float, op: Literal['add', 'sub', 'mul', 'div']) -> float:
    """
    Performs a basic arithmetic operation on two numbers.

    Args:
        firstNumber: the left operand
        secondNumber: the right operand
        op: the operation to perform. Must be exactly one of:
            'add' (+), 'sub' (-), 'mul' (*), 'div' (/)
    """
    if op == 'add':
        return firstNumber + secondNumber
    elif op == 'sub':
        return firstNumber - secondNumber
    elif op == 'mul':
        return firstNumber * secondNumber
    elif op == 'div':
        if secondNumber == 0:
            return "Error: cannot divide by zero."
        return firstNumber / secondNumber
    else:
        # return instead of raise, so the model can see the mistake and retry
        return f"Error: unknown op {op!r}. Use one of: 'add', 'sub', 'mul', 'div'."

In [49]:
tools = [web_search_tool, calculator]

chatModelOld = ChatOpenAI()

chatModel = chatModelOld.bind_tools(tools)

In [50]:
state = StateGraph(ToolsState)



In [51]:
# creating nodes

def chatNode(toolState: ToolsState) -> ToolsState:
    messages = toolState['messages']
    response = chatModel.invoke(messages)
    return {'messages': [response]}


toolNode = ToolNode(tools)

In [52]:
state.add_node('chatNode', chatNode)
state.add_node('toolNode', toolNode)

state.add_edge(START, 'chatNode')

# tools_condition returns the literal 'tools' or END, so map 'tools' -> our node name
state.add_conditional_edges('chatNode', tools_condition, {'tools': 'toolNode', END: END})

state.add_edge('toolNode', 'chatNode')

In [53]:
checkpointer = InMemorySaver()

In [54]:
CONFIG = {"configurable": {"thread_id": 1}}

workflow = state.compile(checkpointer=checkpointer)

ValueError: At 'chatNode' node, 'tools_condition' branch found unknown target 'tools'